In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/oil_price.csv",
    parse_dates=["observation_date"],
    na_values=["."],
)
df = df.rename(columns={"observation_date": "date", "DCOILWTICO": "price"})
df.head()

,date,price
0,2021-10-06,77.66
1,2021-10-07,78.46
2,2021-10-08,79.55
3,2021-10-11,80.64
4,2021-10-12,80.75


In [2]:
print(df["price"].head(3))              # one column
print(df[["date", "price"]].head(3))    # several columns
print(df.iloc[0:3])                     # by position
print(df[df["price"] > 100].head())     # by condition
print(df[(df["date"] >= "2026-01-01") & (df["price"] < 70)].head())  # two conditions

0    77.66
1    78.46
2    79.55
Name: price, dtype: float64
        date  price
0 2021-10-06  77.66
1 2021-10-07  78.46
2 2021-10-08  79.55
        date  price
0 2021-10-06  77.66
1 2021-10-07  78.46
2 2021-10-08  79.55
          date   price
104 2022-03-01  103.66
105 2022-03-02  110.74
106 2022-03-03  107.69
107 2022-03-04  115.77
108 2022-03-07  119.26
           date  price
1107 2026-01-02  57.21
1108 2026-01-05  58.10
1109 2026-01-06  56.97
1110 2026-01-07  56.01
1111 2026-01-08  57.74


In [3]:
s = df.set_index("date")["price"]
print(s["2022-03-01":"2022-03-15"])     # date range
print(s.loc["2025"].describe())         # a whole year

date
2022-03-01    103.66
2022-03-02    110.74
2022-03-03    107.69
2022-03-04    115.77
2022-03-07    119.26
2022-03-08    123.64
2022-03-09    108.81
2022-03-10    105.93
2022-03-11    109.31
2022-03-14    103.22
2022-03-15     96.42
Name: price, dtype: float64
count    248.000000
mean      65.388105
std        5.286754
min       55.440000
25%       61.585000
50%       64.385000
75%       68.737500
max       80.730000
Name: price, dtype: float64


In [4]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["daily_change"] = df["price"].pct_change()
df.head()

,date,price,year,month,daily_change
0,2021-10-06,77.66,2021,10,NaN
1,2021-10-07,78.46,2021,10,0.010301
2,2021-10-08,79.55,2021,10,0.013892
3,2021-10-11,80.64,2021,10,0.013702
4,2021-10-12,80.75,2021,10,0.001364


In [5]:
by_year = df.groupby("year")["price"].agg(["mean", "min", "max", "count"])
print(by_year)

           mean    min     max  count
year                                 
2021  77.312667  65.44   85.64     60
2022  94.902869  71.05  123.64    251
2023  77.576532  66.61   93.67    248
2024  76.632240  66.73   87.69    250
2025  65.388105  55.44   80.73    248
2026  85.510524  56.01  114.58    191


In [6]:
print(df[df["price"].isna()].head())
df["price_ffill"] = df["price"].ffill()
print(df[df["price"].isna()][["date", "price", "price_ffill"]].head())

         date  price  year  month  daily_change
36 2021-11-25    NaN  2021     11           NaN
37 2021-11-26    NaN  2021     11           NaN
57 2021-12-24    NaN  2021     12           NaN
73 2022-01-17    NaN  2022      1           NaN
98 2022-02-21    NaN  2022      2           NaN
         date  price  price_ffill
36 2021-11-25    NaN        78.32
37 2021-11-26    NaN        78.32
57 2021-12-24    NaN        73.89
73 2022-01-17    NaN        83.82
98 2022-02-21    NaN        91.26


In [7]:
weekly = s.resample("W").mean()
monthly = s.resample("ME").mean()
print(weekly.head())
print(monthly.head())
print(len(s), "daily rows ->", len(weekly), "weekly rows")

date
2021-10-10    78.556667
2021-10-17    81.176000
2021-10-24    83.476000
2021-10-31    83.844000
2021-11-07    81.788000
Freq: W-SUN, Name: price, dtype: float64
date
2021-10-31    82.119444
2021-11-30    79.147500
2021-12-31    71.711818
2022-01-31    83.222000
2022-02-28    91.641053
Freq: ME, Name: price, dtype: float64
1305 daily rows -> 262 weekly rows


In [8]:
yearly = df.groupby("year")["price"].mean().reset_index(name="avg_price")

plan = pd.DataFrame({
    "year": [2021, 2022, 2023, 2024, 2025, 2026],
    "plan_assumed_price": [75, 80, 80, 75, 70, 70],
})

merged = yearly.merge(plan, on="year", how="inner")
merged["gap"] = merged["avg_price"] - merged["plan_assumed_price"]
merged["gap_pct"] = (merged["gap"] / merged["plan_assumed_price"] * 100).round(1)
print(merged)

   year  avg_price  plan_assumed_price        gap  gap_pct
0  2021  77.312667                  75   2.312667      3.1
1  2022  94.902869                  80  14.902869     18.6
2  2023  77.576532                  80  -2.423468     -3.0
3  2024  76.632240                  75   1.632240      2.2
4  2025  65.388105                  70  -4.611895     -6.6
5  2026  85.510524                  70  15.510524     22.2
